# Generate Synthetic Employee Profile Data

Last modified: 2026-03-21

This notebook shows one simple pattern for creating **synthetic workforce profile data** from a small seed table.  

What it does:
- generate synthetic identifiers with `Faker`
- sample independent variables from their observed distributions
- preserve grouped distributions for chosen role variables
- generate correlated fields from profile-specific lookup tables
- apply post-processing business rules

In [ ]:
# uncomment this to install Faker
# %pip install faker

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime
from IPython.display import display

In [2]:
def gen_pii_variables(nrows: int, faker_instance: Faker, cnames: list[str]) -> pd.DataFrame:
    data = {}
    for col in cnames:
        if col == "Employee Name":
            data[col] = [faker_instance.name() for _ in range(nrows)]
        else:
            data[col] = [faker_instance.unique.random_number(digits=10, fix_len=True) for _ in range(nrows)]
    return pd.DataFrame(data)

Creates `nrow` timestamps using the date range patterns observed in the seed dataset.

In [3]:
def gen_date_variables(orig_df: pd.DataFrame, num_rows: int, faker_instance: Faker, cnames: list[str]) -> pd.DataFrame:
    date_df = pd.DataFrame(index=range(num_rows))

    for cname in cnames:
        if cname not in orig_df.columns or orig_df[cname].dropna().empty:
            continue

        date_series = pd.to_datetime(orig_df[cname], errors='coerce').dropna()
        year_dist = date_series.dt.year.value_counts(normalize=True)
        month_dist = date_series.dt.month.value_counts(normalize=True)

        synth_years = np.random.choice(year_dist.index, size=num_rows, p=year_dist.values)
        synth_months = np.random.choice(month_dist.index, size=num_rows, p=month_dist.values)

        synth_dates = []
        for year, month in zip(synth_years, synth_months):
            last_day = pd.Timestamp(year=year, month=month, day=1).days_in_month
            synth_date = faker_instance.date_between_dates(
                date_start=datetime(year, month, 1).date(),
                date_end=datetime(year, month, last_day).date()
            )
            synth_dates.append(synth_date)

        date_df[cname] = pd.to_datetime(synth_dates)

    return date_df

Creates a new synthetic table using the subset of **independent** columns. For each column, the notebook computes the empirical probability mass function from the seed table and then samples `nrows` values from that observed distribution. For example, if a role-related variable has values distributed as 60%, 30%, and 10%, the synthetic column is generated with the same proportions.

In [4]:
def gen_independent_variables(orig_df: pd.DataFrame, nrows: int, cnames: list[str]) -> pd.DataFrame:
    synth_df = pd.DataFrame(index=range(nrows))

    for cname in cnames:
        if cname in orig_df.columns:
            cleaned_series = orig_df[cname].dropna()
            if not cleaned_series.empty:
                dist = cleaned_series.value_counts(normalize=True)
                synth_df[cname] = np.random.choice(dist.index, size=nrows, p=dist.values)

    return synth_df

Generates synthetic data for grouping variables while preserving their **combined** distribution, such as `Job Family Code` with `Job Level`.

In [5]:
def gen_grouping_variables(orig_df: pd.DataFrame, nrows: int, cnames: list[str]) -> pd.DataFrame:
    if isinstance(cnames, str):
        cnames = [cnames]

    dist = orig_df[cnames].value_counts(normalize=True)
    indices = np.random.choice(len(dist), size=nrows, p=dist.values)
    synth_groups_df = pd.DataFrame(dist.index[indices].to_list(), columns=cnames)

    return synth_groups_df

Builds a dictionary that maps each unique grouping value (the key) to the observed values in the correlated columns. This becomes a lookup table for the realistic combinations present in the seed dataset.

Example for `Job Family Code`:

```python
{
    'LG-402': {
        'Previous Department': ['Supply Chain Operations'],
        'Professional Training': ['Logistics Management Program']
    },
    'BI-220': {
        'Previous Department': ['Business Intelligence'],
        'Professional Training': ['Strategic Analysis Workshop']
    }
}
```

In [6]:
def create_correlation_profiles(orig_df: pd.DataFrame, grouping_name: str | list[str], cnames: list[str]) -> dict:
    profiles = {}

    if isinstance(grouping_name, str):
        grouping_name = [grouping_name]

    df_cleaned = orig_df.dropna(subset=grouping_name)
    grouped = df_cleaned.groupby(grouping_name)

    for key, group_df in grouped:
        profiles[key] = {}
        for cname in cnames:
            if cname in group_df:
                unique_values = group_df[cname].dropna().unique().tolist()
                profiles[key][cname] = unique_values

    return profiles

The correlated variables are then generated from the profile dictionary returned by `create_correlation_profiles()`. For each key in the grouping columns (here, the `Job Family Code` and `Job Level` combination), the notebook randomly selects values from the observed correlated options for that profile. This lets the synthetic dataset keep plausible role-to-training, role-to-title, and role-to-department relationships.

In [7]:
def gen_correlated_variables(pkey_series: pd.DataFrame, profiles: dict, cnames: list[str]) -> pd.DataFrame:
    new_data = {cname: [None] * len(pkey_series) for cname in cnames}

    # Handle either a single grouping column (Series) or multiple grouping columns (DataFrame).
    if isinstance(pkey_series, pd.Series):
        for i, key_val in enumerate(pkey_series):
            profile = profiles.get(key_val)
            if profile:
                for cname in cnames:
                    possible_values = profile.get(cname)
                    if possible_values:
                        new_data[cname][i] = np.random.choice(possible_values)
    else:
        for i, key_vals in enumerate(pkey_series.itertuples(index=False, name=None)):
            profile = profiles.get(key_vals)
            if profile:
                for cname in cnames:
                    possible_values = profile.get(cname)
                    if possible_values:
                        new_data[cname][i] = np.random.choice(possible_values)

    return pd.DataFrame(new_data)

Applies a few post-processing business rules to correct obvious inconsistencies.

In [8]:
def apply_business_rules(df: pd.DataFrame) -> pd.DataFrame:
    # Rule 1:
    # Management and director levels should have at least a Bachelor's degree.
    leadership_levels = df['Job Level'].isin(['M1', 'M2', 'M3', 'D1', 'D2', 'D3'])
    no_degree = ~df['Education Level'].isin(['Bachelors', 'Masters'])

    for index in df[leadership_levels & no_degree].index:
        df.loc[index, 'Education Level'] = np.random.choice(['Bachelors', 'Masters'], p=[0.8, 0.2])

    # Rule 2:
    # If the education level is high school only, use "N/A" as the field of study.
    hs_mask = df['Education Level'].eq('High School')
    df.loc[hs_mask, 'Field of Study'] = 'N/A'

    return df

Some configuration values.

In [9]:
Faker.seed(1984)
faker_instance = Faker()

nrow_synth = 100  # number of synthetic rows to generate

Proposed dataset features. If a variable is requested during synthesis but does not exist in the seed dataset, the resulting synthetic values for that field will be `None`.

In [10]:
PII_cols = ['Employee Name', 'Employee ID', 'Internal Record ID']
independent_cols = [
    'Education Level', 'Field of Study', 'Marital Status', 'Number of Dependents',
    'Dual-Career Household', 'Remote Eligible', 'Employment Type', 'Additional Language'
]
timestamp_cols = ['Promotion Date', 'Assignment End Date']
group_vars_profile_corr = ['Job Family Code', 'Job Level']
correlated_cols = [
    'Current Role Code', 'Secondary Skill Code', 'Professional Training',
    'Previous Department', 'Current Job Title', 'Current Department'
]

Dictionary below acts as a small **seed dataset** which is artificially generated. In practice, replace this with a CSV, Excel file, Parquet file, or database extract from your own environment.

In [11]:
my_data = {
    'Internal Record ID': [1000030283, 1000023760, 1000045678, 1000098765, 1000012345, 1000054321, 1000067890, 1000011223, 1000033445, 
                           1000055667, 1000077889, 1000099001, 1000024680, 1000013579, 1000086420, 1000025814, 1000036925, 1000047136, 
                           1000058247, 1000069358, 1000071469, 1000082570, 1000093681, 1000015792, 1000026803],
    'Employee Name': ['Employee 01', 'Employee 02', 'Employee 03', 'Employee 04', 'Employee 05', 'Employee 06', 'Employee 07', 'Employee 08', 
                      'Employee 09', 'Employee 10', 'Employee 11', 'Employee 12', 'Employee 13', 'Employee 14', 'Employee 15', 'Employee 16', 
                      'Employee 17', 'Employee 18', 'Employee 19', 'Employee 20', 'Employee 21', 'Employee 22', 'Employee 23', 'Employee 24', 
                      'Employee 25'],
    'Employee ID': [1186112977, 1297888060, 1345678912, 1456789123, 1567891234, 1678912345, 1789123456, 1891234567, 1912345678, 1123456789, 
                    1234567890, 1357924680, 1468013579, 1579135791, 1680246802, 1791357913, 1802468024, 1913579135, 1135792468, 1246801357, 
                    1368024680, 1479135791, 1580246802, 1691357913, 1702468024],
    'Job Level': ['D1', 'M1', 'D2', 'L4', 'L3', 'P3', 'M2', 'L2', 'P2', 'M3', 'P1', 'L1', 'D3', 'L3', 'M1', 'S1', 'D1', 'L4', 'L3', 'M2', 
                  'P3', 'M1', 'D2', 'L2', 'M3'],
    'Education Level': ['Masters', 'Bachelors', 'Masters', 'Some College', 'High School', 'Bachelors', 'Bachelors', 'High School', 
                        'Bachelors', 'Masters', 'Bachelors', 'High School', 'Masters', 'Some College', 'Bachelors', 'Associates', 
                        'Bachelors', 'Some College', 'High School', 'Bachelors', 'Bachelors', 'Some College', 'Masters', 'High School', 
                        'High School'],
    'Field of Study': ['Logistics Management', 'Criminal Justice', 'Business Administration', 'General Studies', 'N/A', 'Political Science', 
                       'Engineering', 'N/A', 'History', 'Strategic Studies', 'General Studies', 'N/A', 'International Relations', 'N/A', 
                       'History', 'Electronics Technology', 'Finance', 'General Studies', 'N/A', 'Public Administration', 'Aeronautical Science', 
                       'N/A', 'Systems Engineering', 'N/A', 'N/A'],
    'Marital Status': ['Married', 'Married', 'Married', 'Single', 'Single', 'Single', 'Married', 'Single', 'Single', 'Married', 'Single', 
                       'Single', 'Married', 'Single', 'Married', 'Married', 'Single', 'Single', 'Single', 'Married', 'Single', 'Married', 
                       'Single', 'Single', 'Married'],
    'Number of Dependents': [2, 3, 2, 0, 0, 0, 4, 0, 0, 3, 0, 0, 3, 0, 1, 2, 0, 0, 0, 2, 0, 2, 0, 0, 5],
    'Dual-Career Household': ['No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 
                              'No', 'No', 'No', 'No', 'No', 'No', 'No'],
    'Remote Eligible': ['No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 
                        'No', 'No', 'No', 'No', 'No', 'No'],
    'Employment Type': ['Full-Time', 'Full-Time', 'Full-Time', 'Full-Time', 'Full-Time', 'Full-Time', 'Full-Time', 'Full-Time', 'Full-Time', 
                        'Full-Time', 'Full-Time', 'Full-Time', 'Full-Time', 'Part-Time', 'Full-Time', 'Full-Time', 'Full-Time', 'Full-Time', 
                        'Full-Time', 'Full-Time', 'Full-Time', 'Part-Time', 'Full-Time', 'Full-Time', 'Full-Time'],
    'Additional Language': ['Spanish', 'Japanese', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'Spanish', 'None', 'None', 'None', 
                            'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'Polish', 'None', 'Italian', 'None'],
    'Job Family Code': ['LG-402', 'MG-999', 'PM-320', 'FO-531', 'OP-311', 'BI-220', 'FE-811', 'PC-343', 'PD-765', 'IT-699', 'PM-320', 'OP-311', 
                        'ST-841', 'DC-312', 'EN-371', 'TR-360', 'FN-404', 'DA-231', 'OT-331', 'HR-110', 'PD-763', 'TL-369', 'EM-802', 'OT-341', 
                        'SO-581'],
    'Promotion Date': ['10/1/2022', '3/1/2021', '8/1/2023', '5/1/2020', '11/1/2019', '7/1/2023', '2/1/2022', '12/1/2023', '6/1/2024', 
                       '4/1/2020', '5/28/2025', '9/1/2024', '7/1/2021', '1/1/2021', '10/1/2019', '9/1/2023', '11/1/2021', '4/1/2022', 
                       '3/1/2023', '6/1/2018', '12/1/2022', '8/1/2020', '9/1/2020', '7/1/2022', '1/1/2017'],
    'Assignment End Date': ['10/1/2028', '3/1/2027', '8/1/2030', '5/1/2026', '11/1/2025', '7/1/2029', '2/1/2028', '12/1/2027', '6/1/2028', 
                            '4/1/2026', '5/28/2029', '9/1/2028', '7/1/2029', '1/1/2027', '10/1/2025', '9/1/2029', '11/1/2027', '4/1/2028', 
                            '3/1/2027', '6/1/2024', '12/1/2028', '8/1/2026', '9/1/2026', '7/1/2026', '1/1/2025'],
    'Current Role Code': ['LG-402', 'MG-999', 'PM-320', 'FO-531', 'OP-311', 'BI-220', 'FE-811', 'PC-343', 'PD-765', 'IT-699', 'PM-320', 
                          'OP-311', 'ST-841', 'DC-312', 'EN-371', 'TR-360', 'FN-404', 'DA-231', 'OT-331', 'HR-110', 'PD-763', 'TL-369', 
                          'EM-802', 'OT-341', 'SO-581'],
    'Secondary Skill Code': ['None', 'TL-369', 'None', 'None', 'None', 'DA-231', 'None', 'None', 'None', 'IT-629', 'None', 'None', 'PM-320', 
                             'None', 'None', 'OP-311', 'None', 'None', 'None', 'None', 'None', 'MG-999', 'None', 'None', 'None'],
    'Professional Training': ['Logistics Management Program', 'Operations Leadership Seminar', 'Program Leadership Bootcamp', 
                              'Supervisor Development Program', 'Frontline Supervisor Workshop', 'Strategic Analysis Workshop', 
                              'Field Operations Leadership', 'Early Career Foundations', 'Leadership Foundations', 'IT Operations Leadership', 
                              'Leadership Foundations', 'Operations Onboarding', 'Executive Strategy Seminar', 'Supervisor Development Program', 
                              'Project Execution Workshop', 'Technical Trainer Certification', 'Financial Management Certificate', 
                              'Data Analysis Fundamentals', 'Team Lead Essentials', 'HR Systems Fundamentals', 'Leadership Foundations', 
                              'Frontline Leadership Program', 'Engineering Management Basics', 'Frontline Supervisor Workshop', 
                              'Corporate Investigations Program'],
    'Previous Department': ['Supply Chain Operations', 'Regional Operations South', 'Program Delivery West', 'Transportation Services', 
                            'Service Delivery West', 'Business Intelligence', 'Field Engineering West', 'Central Operations Support', 
                            'Advanced Product Team', 'Infrastructure Services', 'Program Delivery East', 'Service Delivery South', 
                            'Strategy & Planning', 'Security Compliance', 'Facilities Projects', 'Learning & Development', 'Finance Office', 
                            'Analytics Center', 'Site Operations North', 'People Operations', 'Product Development Lab', 'Field Operations Pacific', 
                            'Field Engineering East', 'Site Operations Central', 'Security Operations'],
    'Current Job Title': ['Logistics Manager', 'Operations Team Manager', 'Program Manager', 'Fleet Operations Lead', 'Operations Specialist', 
                          'Business Intelligence Manager', 'Field Engineering Supervisor', 'Procurement Coordinator', 
                          'Product Development Associate', 'IT Operations Lead', 'Associate Program Manager', 'Operations Specialist', 
                          'Strategy Operations Director', 'Distribution Coordinator', 'Facilities Engineer', 'Training Program Lead', 
                          'Finance Manager', 'Data Analyst', 'Operations Technician', 'HR Business Partner', 'Associate Product Developer', 
                          'Team Lead', 'Engineering Program Manager', 'Operations Technician', 'Security Operations Director'],
    'Current Department': ['Supply Chain Operations', 'Regional Operations South', 'Program Delivery West', 'Transportation Services', 
                           'Service Delivery West', 'Business Intelligence', 'Field Engineering West', 'Central Operations Support', 
                           'Advanced Product Team', 'Infrastructure Services', 'Program Delivery East', 'Service Delivery South', 
                           'Strategy & Planning', 'Security Operations', 'Facilities Projects', 'Learning & Development', 'Finance Office', 
                           'Analytics Center', 'Site Operations North', 'People Operations', 'Product Development Lab', 
                           'Field Operations Pacific', 'Field Engineering East', 'Site Operations Central', 'Executive Operations']
}

df = pd.DataFrame(my_data)
df.head()

,Internal Record ID,Employee Name,Employee ID,Job Level,Education Level,Field of Study,Marital Status,Number of Dependents,Dual-Career Household,Remote Eligible,...,Additional Language,Job Family Code,Promotion Date,Assignment End Date,Current Role Code,Secondary Skill Code,Professional Training,Previous Department,Current Job Title,Current Department
0,1000030283,Employee 01,1186112977,D1,Masters,Logistics Management,Married,2,No,No,...,Spanish,LG-402,10/1/2022,10/1/2028,LG-402,None,Logistics Management Program,Supply Chain Operations,Logistics Manager,Supply Chain Operations
1,1000023760,Employee 02,1297888060,M1,Bachelors,Criminal Justice,Married,3,Yes,Yes,...,Japanese,MG-999,3/1/2021,3/1/2027,MG-999,TL-369,Operations Leadership Seminar,Regional Operations South,Operations Team Manager,Regional Operations South
2,1000045678,Employee 03,1345678912,D2,Masters,Business Administration,Married,2,No,No,...,None,PM-320,8/1/2023,8/1/2030,PM-320,None,Program Leadership Bootcamp,Program Delivery West,Program Manager,Program Delivery West
3,1000098765,Employee 04,1456789123,L4,Some College,General Studies,Single,0,No,No,...,None,FO-531,5/1/2020,5/1/2026,FO-531,None,Supervisor Development Program,Transportation Services,Fleet Operations Lead,Transportation Services
4,1000012345,Employee 05,1567891234,L3,High School,N/A,Single,0,No,No,...,None,OP-311,11/1/2019,11/1/2025,OP-311,None,Frontline Supervisor Workshop,Service Delivery West,Operations Specialist,Service Delivery West


In [12]:
pii_df = gen_pii_variables(nrow_synth, faker_instance, PII_cols)

In [13]:
date_df = gen_date_variables(df, nrow_synth, faker_instance, timestamp_cols)

In [14]:
grouping_df = gen_grouping_variables(df, nrow_synth, group_vars_profile_corr)

In [15]:
independent_df = gen_independent_variables(df, nrow_synth, independent_cols)

In [16]:
correlation_profiles = create_correlation_profiles(df, group_vars_profile_corr, correlated_cols)

In [17]:
correlated_df = gen_correlated_variables(grouping_df, correlation_profiles, correlated_cols)

In [18]:
final_synth_df = pd.concat([pii_df, grouping_df, independent_df, date_df, correlated_df], axis=1)
final_synth_df_cleaned = apply_business_rules(final_synth_df)

print("Number of rows:", len(final_synth_df_cleaned), "\n")
display(final_synth_df_cleaned)

Number of rows: 100 



,Employee Name,Employee ID,Internal Record ID,Job Family Code,Job Level,Education Level,Field of Study,Marital Status,Number of Dependents,Dual-Career Household,...,Employment Type,Additional Language,Promotion Date,Assignment End Date,Current Role Code,Secondary Skill Code,Professional Training,Previous Department,Current Job Title,Current Department
0,Johnny Morgan,4293571030,9166268625,FE-811,M2,Bachelors,N/A,Single,0,No,...,Full-Time,None,2020-12-10,2026-05-03,FE-811,None,Field Operations Leadership,Field Engineering West,Field Engineering Supervisor,Field Engineering West
1,Scott Atkinson,8854500828,5701287340,PC-343,L2,Some College,N/A,Single,2,No,...,Full-Time,None,2024-06-25,2030-11-06,PC-343,None,Early Career Foundations,Central Operations Support,Procurement Coordinator,Central Operations Support
2,David Alvarado,3848904512,7655913056,BI-220,P3,Bachelors,Logistics Management,Married,3,No,...,Full-Time,None,2023-06-16,2028-06-11,BI-220,DA-231,Strategic Analysis Workshop,Business Intelligence,Business Intelligence Manager,Business Intelligence
3,Steven Fox,4212883238,7449308349,OP-311,L1,Masters,Electronics Technology,Married,2,No,...,Full-Time,None,2021-03-16,2028-10-13,OP-311,None,Operations Onboarding,Service Delivery South,Operations Specialist,Service Delivery South
4,Emma Lee,6022351033,5771332349,OP-311,L1,High School,N/A,Single,2,No,...,Full-Time,Japanese,2023-12-27,2029-08-14,OP-311,None,Operations Onboarding,Service Delivery South,Operations Specialist,Service Delivery South
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Mary Cox,8243222098,9707513194,DA-231,L4,High School,N/A,Single,0,Yes,...,Full-Time,None,2023-06-04,2027-10-20,DA-231,None,Data Analysis Fundamentals,Analytics Center,Data Analyst,Analytics Center
96,Margaret Arellano,2313113474,6780344839,OP-311,L3,Some College,Business Administration,Married,1,No,...,Full-Time,None,2020-09-24,2028-02-07,OP-311,None,Frontline Supervisor Workshop,Service Delivery West,Operations Specialist,Service Delivery West
97,Arthur Mcdowell,4998295962,3978003012,FE-811,M2,Masters,General Studies,Married,4,No,...,Full-Time,Spanish,2024-06-18,2028-07-26,FE-811,None,Field Operations Leadership,Field Engineering West,Field Engineering Supervisor,Field Engineering West
98,William Jimenez,8031031742,9978673646,OT-341,L2,Bachelors,N/A,Married,0,No,...,Full-Time,None,2025-04-14,2026-06-02,OT-341,None,Frontline Supervisor Workshop,Site Operations Central,Operations Technician,Site Operations Central


In [19]:
# final_synth_df_cleaned.to_csv("synthetic_employee_profiles.csv", index=False)